# Round 0 detector on Kaggle

Generates the 8,000 frame dataset with the `base` domain randomization preset, trains YOLO11n-pose with the `base` training preset (at most 20 GPU minutes), and exports `detector.onnx`.

Settings: Internet on, accelerator GPU T4. When it finishes, download `/kaggle/working/round0.zip` from the Output panel.

In [ ]:
import time

notebook_start = time.time()
REPO_URL = "https://github.com/Ramykaz/kestrel"
W = "/kaggle/working"
DATA = f"{W}/datasets/r00-base"
RUN = f"{W}/train/r00"

In [ ]:
!git clone --depth 1 $REPO_URL $W/kestrel
!pip install -q uv

In [ ]:
%cd $W/kestrel
!uv sync --locked --no-dev --extra train
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!.venv/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## Dataset

In [ ]:
!.venv/bin/kestrel gen --out $DATA --frames 8000 --dr base --seed 0 --workers 4
!.venv/bin/kestrel data preview $DATA --out $W/contact_sheet.png

In [ ]:
from IPython.display import Image

Image(f"{W}/contact_sheet.png", width=1200)

## Training

In [ ]:
!.venv/bin/kestrel train --dataset $DATA --out $RUN --preset base --device 0

In [ ]:
!mkdir -p $W/round0
!cp $RUN/detector.onnx $RUN/train_report.json $W/contact_sheet.png $W/round0/
!cp $DATA/manifest.json $W/round0/dataset_manifest.json
!cd $W && zip -r -q round0.zip round0 && ls -la round0 round0.zip
!cat $W/round0/train_report.json
print(f"notebook time: {(time.time() - notebook_start) / 60:.1f} min")